# 第5回：何を、いつ、何のために予測するか

**今日の問い：モデル構築より前に決めるべきことは何か。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 利用者・判断・予測時点を1文にする
- 目的変数と利用可能な説明変数を分け、リーク候補を自動監査する
- 誤りのコストから期待値を計算し、指標と閾値を業務要件で決める

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 予測時点：モデルを実際に使う瞬間
- ベースライン：複雑なモデルと比較する単純な基準
- コスト行列：誤りの種類ごとの損失をまとめた表
- 期待コスト：確率×損失で見積もる平均的な損失
- リーク監査：目的変数と強く結びつく怪しい列を洗い出す確認

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## 予測問題を1文にする

例：**実験条件を決める時点で使える情報から収率を予測し、優先して実施する条件を選ぶ。**

`post_assay_signal`・`purity_pct`・`yield_pct`は実験後の値なので、この時点の説明変数にはできません。


In [ ]:
available_at_planning = [
    "scaffold_group", "solvent", "catalyst", "temperature_c", "reaction_time_h",
    "concentration_m", "molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds",
]
unavailable_at_planning = ["yield_pct", "active", "post_assay_signal", "purity_pct"]
print("計画時に使える列:", available_at_planning)
print("実験後に得られる列:", unavailable_at_planning)


## TRY：単純な予測を基準にする


In [ ]:
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.model_selection import train_test_split

train, valid = train_test_split(df, test_size=0.25, random_state=42)
reg = DummyRegressor(strategy="mean").fit(train[["molecular_weight"]], train["yield_pct"])
cls = DummyClassifier(strategy="most_frequent").fit(train[["molecular_weight"]], train["active"])
print("平均収率だけのMAE:", round(mean_absolute_error(valid["yield_pct"], reg.predict(valid[["molecular_weight"]])), 2))
print("多数派だけの正解率:", round(accuracy_score(valid["active"], cls.predict(valid[["molecular_weight"]])), 3))


## CORE深掘り：リーク候補を自動で洗い出す

目的変数と極端に相関する列は、測定後情報が紛れ込んでいないか疑います。監査を関数にしておくと、自社データでも使い回せます。


In [ ]:
def leakage_audit(frame, target: str, threshold: float = 0.9):
    "目的変数と相関が極端に高い数値列を、リーク候補として洗い出す。"
    numeric = frame.select_dtypes(include="number")
    corr = numeric.corrwith(frame[target]).abs().drop(labels=[target], errors="ignore")
    report = corr.sort_values(ascending=False).to_frame("|相関|")
    report["リーク候補"] = report["|相関|"] >= threshold
    return report.round(3)

display(leakage_audit(df, target="active", threshold=0.6))
print("post_assay_signalは測定後の値。相関が高くても計画時には使えない。")


## TRY：自分のテーマを整理する

利用者・判断・予測時点・目的変数・使える列・使えない列・回帰/分類・単純基準の8点を1枚に書きます。

## ASK COPILOT

曖昧な点は推測で埋めず、確認質問として返すよう依頼します。


## DEEP DIVE：コスト行列から最適な閾値を決める

最適な閾値は指標ではなく、誤りのコストで決まります。見逃し（偽陰性）が高くつく状況を想定します。


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

feat = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_tr, X_te, y_tr, y_te = train_test_split(df[feat], df["active"], test_size=0.3, random_state=42, stratify=df["active"])
clf = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

cost_fn, cost_fp = 10, 1  # 見逃し=有望条件を逃す損失、偽陽性=無駄な追試
rows = []
for t in np.linspace(0.1, 0.9, 17):
    pred = (proba >= t).astype(int)
    fp = int(((pred == 1) & (y_te == 0)).sum())
    fn = int(((pred == 0) & (y_te == 1)).sum())
    rows.append({"閾値": round(t, 2), "偽陽性": fp, "偽陰性": fn, "期待コスト": fp * cost_fp + fn * cost_fn})
cost_table = pd.DataFrame(rows)
best = cost_table.loc[cost_table["期待コスト"].idxmin(), "閾値"]
display(cost_table)
print("コスト最小の閾値:", best, " / 見逃しが高いほど閾値は下がる")


### 指標は意思決定から逆算する

見逃しを避けたい探索段階ならrecall寄り、追試コストが高い絞り込み段階ならprecision寄り。「良いスコア」ではなく「どう使うか」で選びます。


## よくある誤り

- 入手できる列をすべて使う
- 目的変数が測定や運用で不安定
- 精度目標だけで利用方法とコストが決まっていない

## SELF-STUDY（任意・30〜60分）

- 自社テーマを機密情報なしで問題設定キャンバスへ落とす
- 偽陽性・偽陰性のコストを入れ、期待コスト最小の閾値を計算する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 誰が何を判断するモデルか
2. 予測時点で本当に得られる列はどれか
3. コスト行列から最適な閾値をどう求めるか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
